# Feature Extraction: Creating New Features

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/03-feature-engineering/02_feature_extraction.ipynb)

## Objectives
- Understand feature extraction vs feature engineering
- Learn to create derived features from raw data
- Build interaction features
- Extract domain-specific features
- Implement aggregate features

## 1. Feature Extraction vs Engineering

**Feature Extraction**: Creating new features from existing raw data
- Ratios and proportions
- Sums and differences
- Products and divisions
- Domain-specific calculations

**Feature Engineering**: Broader term including:
- Extraction, transformation, selection
- Encoding, scaling, normalization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)
sns.set_theme()

print("✅ Libraries loaded")

## 2. Create Sample Dataset

E-commerce customer purchase dataset

In [ ]:
# Create realistic e-commerce dataset
np.random.seed(42)
n_samples = 500

df = pd.DataFrame({
    'product_price': np.random.uniform(10, 500, n_samples),
    'quantity': np.random.randint(1, 20, n_samples),
    'customer_age': np.random.randint(18, 80, n_samples),
    'previous_purchases': np.random.randint(0, 50, n_samples),
    'customer_lifetime_value': np.random.uniform(0, 5000, n_samples),
    'discount_applied': np.random.choice([0, 0.05, 0.10, 0.15, 0.20], n_samples),
    'shipping_cost': np.random.uniform(5, 50, n_samples),
    'days_since_signup': np.random.randint(1, 3650, n_samples)
})

# Create target (1 if high-value customer, 0 otherwise)
df['high_value'] = ((df['customer_lifetime_value'] > 2000) | 
                     (df['previous_purchases'] > 20)).astype(int)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head(10))
print(f"\nBasic statistics:")
print(df.describe())

## 3. Ratio Features

Create meaningful ratios from numeric features

In [ ]:
# Create ratio features
df['total_order_value'] = df['product_price'] * df['quantity']
df['effective_price'] = df['product_price'] * (1 - df['discount_applied'])
df['price_to_clv_ratio'] = df['product_price'] / (df['customer_lifetime_value'] + 1)  # +1 to avoid division by zero
df['shipping_percentage'] = (df['shipping_cost'] / df['total_order_value']) * 100
df['purchase_frequency'] = df['previous_purchases'] / (df['days_since_signup'] / 365 + 0.1)  # purchases per year
df['avg_purchase_value'] = df['customer_lifetime_value'] / (df['previous_purchases'] + 1)

print("📊 Ratio Features Created:")
print(df[['total_order_value', 'effective_price', 'price_to_clv_ratio', 
          'shipping_percentage', 'purchase_frequency', 'avg_purchase_value']].head(10))

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

features_to_plot = ['total_order_value', 'effective_price', 'price_to_clv_ratio',
                    'shipping_percentage', 'purchase_frequency', 'avg_purchase_value']

for idx, feature in enumerate(features_to_plot):
    axes[idx].hist(df[feature], bins=30, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{feature}')
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 4. Interaction Features

Capture relationships between features

In [ ]:
# Interaction features (multiply or combine related features)
df['age_purchases_interaction'] = df['customer_age'] * df['previous_purchases']
df['price_quantity_interaction'] = df['product_price'] * df['quantity']
df['clv_purchase_freq_interaction'] = df['customer_lifetime_value'] * df['purchase_frequency']
df['young_bulk_buyer'] = ((df['customer_age'] < 35) & (df['quantity'] > 10)).astype(int)
df['loyal_high_value'] = ((df['previous_purchases'] > 15) & (df['customer_lifetime_value'] > 1500)).astype(int)

print("📊 Interaction Features Created:")
print(df[['customer_age', 'previous_purchases', 'age_purchases_interaction',
          'young_bulk_buyer', 'loyal_high_value']].head(10))

# Correlation with target
feature_corr = df[['age_purchases_interaction', 'price_quantity_interaction', 
                    'clv_purchase_freq_interaction', 'young_bulk_buyer', 
                    'loyal_high_value', 'high_value']].corr()['high_value'].drop('high_value')

print(f"\n📊 Interaction Features Correlation with Target:")
print(feature_corr.sort_values(ascending=False))

## 5. Polynomial and Power Features

Create non-linear transformations

In [ ]:
# Power features
df['quantity_squared'] = df['quantity'] ** 2
df['quantity_sqrt'] = np.sqrt(df['quantity'])
df['price_squared'] = df['product_price'] ** 2
df['age_squared'] = df['customer_age'] ** 2
df['clv_log'] = np.log1p(df['customer_lifetime_value'])  # log1p handles 0 values
df['purchases_log'] = np.log1p(df['previous_purchases'])

print("📊 Power Features Created:")
print(df[['quantity', 'quantity_squared', 'quantity_sqrt',
          'customer_lifetime_value', 'clv_log']].head(10))

# Visualize polynomial transformation
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(df['quantity'], df['quantity_squared'], alpha=0.6)
axes[0].set_xlabel('Quantity')
axes[0].set_ylabel('Quantity Squared')
axes[0].set_title('Quadratic Transformation')

axes[1].scatter(df['quantity'], df['quantity_sqrt'], alpha=0.6, color='green')
axes[1].set_xlabel('Quantity')
axes[1].set_ylabel('Quantity Sqrt')
axes[1].set_title('Square Root Transformation')

axes[2].scatter(df['customer_lifetime_value'], df['clv_log'], alpha=0.6, color='red')
axes[2].set_xlabel('CLV (Original)')
axes[2].set_ylabel('CLV (Log)')
axes[2].set_title('Log Transformation')

plt.tight_layout()
plt.show()

## 6. Binning and Categorization

Extract categorical features from continuous variables

In [ ]:
# Age groups
df['age_group'] = pd.cut(df['customer_age'], 
                          bins=[0, 25, 35, 50, 65, 100],
                          labels=['18-25', '26-35', '36-50', '51-65', '65+'])

# Customer value tiers
df['clv_tier'] = pd.qcut(df['customer_lifetime_value'],
                          q=4,
                          labels=['Low', 'Medium', 'High', 'Very High'],
                          duplicates='drop')

# Purchase frequency categories
df['purchase_category'] = pd.cut(df['previous_purchases'],
                                 bins=[-1, 0, 10, 20, 100],
                                 labels=['Never', 'Occasional', 'Regular', 'Frequent'])

print("📊 Categorical Features Created:")
print(f"Age groups distribution:")
print(df['age_group'].value_counts())
print(f"\nCLV tier distribution:")
print(df['clv_tier'].value_counts())
print(f"\nPurchase category distribution:")
print(df['purchase_category'].value_counts())

## 7. Aggregate Features

Summary statistics and aggregations

In [ ]:
# Aggregate features per age group
age_group_stats = df.groupby('age_group').agg({
    'customer_lifetime_value': ['mean', 'median', 'std'],
    'product_price': ['mean', 'max'],
    'previous_purchases': ['mean', 'sum']
}).round(2)

print("📊 Aggregate Statistics by Age Group:")
print(age_group_stats)

# Map aggregates back to individual rows
age_group_mean_clv = df.groupby('age_group')['customer_lifetime_value'].transform('mean')
df['age_group_avg_clv'] = age_group_mean_clv
df['clv_vs_age_group'] = df['customer_lifetime_value'] / df['age_group_avg_clv']

print(f"\n📊 New aggregated features:")
print(df[['age_group', 'customer_lifetime_value', 'age_group_avg_clv', 'clv_vs_age_group']].head(10))

## 8. Feature Importance Before and After Extraction

Impact of extracted features on model performance

# Original features only
original_features = ['product_price', 'quantity', 'customer_age', 
                    'previous_purchases', 'customer_lifetime_value',
                    'discount_applied', 'shipping_cost', 'days_since_signup']

# All extracted features
extracted_features = original_features + [
    'total_order_value', 'effective_price', 'price_to_clv_ratio',
    'shipping_percentage', 'purchase_frequency', 'avg_purchase_value',
    'age_purchases_interaction', 'price_quantity_interaction',
    'clv_purchase_freq_interaction', 'quantity_squared', 'quantity_sqrt',
    'price_squared', 'age_squared', 'clv_log', 'purchases_log'
]

# Train test split
X_train_orig, X_test_orig, y_train, y_test = train_test_split(
    df[original_features], df['high_value'], test_size=0.2, random_state=42
)

X_train_ext, X_test_ext, _, _ = train_test_split(
    df[extracted_features], df['high_value'], test_size=0.2, random_state=42
)

# Train models
rf_orig = RandomForestClassifier(n_estimators=100, random_state=42)
rf_orig.fit(X_train_orig, y_train)
acc_orig = accuracy_score(y_test, rf_orig.predict(X_test_orig))

rf_ext = RandomForestClassifier(n_estimators=100, random_state=42)
rf_ext.fit(X_train_ext, y_train)
acc_ext = accuracy_score(y_test, rf_ext.predict(X_test_ext))

print("📊 Model Performance Comparison:")
print(f"Original Features Only: {acc_orig:.4f}")
print(f"With Extracted Features: {acc_ext:.4f}")
print(f"Improvement: {(acc_ext - acc_orig) * 100:.2f}%")

# Feature importances
feat_imp_ext = pd.DataFrame({
    'Feature': extracted_features,
    'Importance': rf_ext.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\n📊 Top 15 Features by Importance (with extracted):")
print(feat_imp_ext.head(15))

# Visualization
plt.figure(figsize=(12, 6))
plt.barh(range(15), feat_imp_ext['Importance'].head(15).values, color=['red' if f in original_features else 'green' for f in feat_imp_ext['Feature'].head(15)])
plt.yticks(range(15), feat_imp_ext['Feature'].head(15).values)
plt.xlabel('Importance')
plt.title('Feature Importances (Red=Original, Green=Extracted)')
plt.tight_layout()
plt.show()

## 9. Domain-Specific Feature Extraction

Example: Creating business intelligence features

# RFM Analysis (Recency, Frequency, Monetary)
df['recency_score'] = 1 / (df['days_since_signup'] / 365 + 1)  # Higher = more recent
df['frequency_score'] = df['previous_purchases'] / df['previous_purchases'].max()
df['monetary_score'] = df['customer_lifetime_value'] / df['customer_lifetime_value'].max()

# RFM combined score
df['rfm_score'] = (df['recency_score'] + df['frequency_score'] + df['monetary_score']) / 3

# Customer value indicator
df['high_profit_potential'] = ((df['rfm_score'] > 0.5) & (df['customer_age'] < 50)).astype(int)

print("📊 RFM Analysis Features:")
print(df[['days_since_signup', 'recency_score', 'frequency_score', 
          'monetary_score', 'rfm_score', 'high_profit_potential']].head(10))

# Visualize RFM scores
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(df['recency_score'], bins=30, edgecolor='black', color='skyblue')
axes[0, 0].set_title('Recency Score Distribution')

axes[0, 1].hist(df['frequency_score'], bins=30, edgecolor='black', color='lightgreen')
axes[0, 1].set_title('Frequency Score Distribution')

axes[1, 0].hist(df['monetary_score'], bins=30, edgecolor='black', color='coral')
axes[1, 0].set_title('Monetary Score Distribution')

axes[1, 1].hist(df['rfm_score'], bins=30, edgecolor='black', color='gold')
axes[1, 1].set_title('Combined RFM Score Distribution')

plt.tight_layout()
plt.show()

## 10. Feature Creation Pipeline

Reusable function for feature extraction

In [ ]:
def extract_features(df):
    """Extract domain-specific features from raw data"""
    df = df.copy()
    
    # Ratio features
    df['total_order_value'] = df['product_price'] * df['quantity']
    df['effective_price'] = df['product_price'] * (1 - df['discount_applied'])
    df['shipping_percentage'] = (df['shipping_cost'] / df['total_order_value']) * 100
    
    # Interaction features
    df['price_quantity_interaction'] = df['product_price'] * df['quantity']
    
    # Power features
    df['clv_log'] = np.log1p(df['customer_lifetime_value'])
    df['purchases_log'] = np.log1p(df['previous_purchases'])
    
    # RFM features
    df['recency_score'] = 1 / (df['days_since_signup'] / 365 + 1)
    df['frequency_score'] = df['previous_purchases'] / (df['previous_purchases'].max() + 1)
    df['monetary_score'] = df['customer_lifetime_value'] / (df['customer_lifetime_value'].max() + 1)
    df['rfm_score'] = (df['recency_score'] + df['frequency_score'] + df['monetary_score']) / 3
    
    return df

# Apply pipeline
df_engineered = extract_features(df)

print("📊 Features before extraction:", len(df.columns))
print("📊 Features after extraction:", len(df_engineered.columns))
print(f"\n✅ {len(df_engineered.columns) - len(df.columns)} new features created!")

## 11. Best Practices & Common Mistakes

**✅ Best Practices:**
1. Create features based on domain knowledge
2. Monitor for data leakage (train/test contamination)
3. Document feature creation logic
4. Handle edge cases (division by zero, log of negative numbers)
5. Test feature importance on held-out data
6. Create features before train-test split

**❌ Common Mistakes:**
1. Over-extracting too many features (curse of dimensionality)
2. Not handling missing/edge values (NaN, inf)
3. Using test set when fitting feature transformations
4. Creating correlated features without understanding interactions
5. Not validating if new features improve model performance

print("""\n📚 KEY TAKEAWAYS:

Feature Extraction Techniques:
1. Ratio Features: Proportions, rates, percentages
2. Interaction Features: Multiplying, combining features
3. Polynomial Features: Squared, sqrt, log transformations  
4. Domain Features: Business logic (RFM, customer segments)
5. Aggregate Features: Summary statistics, group aggregations

✅ Extracted features often improve model performance
✅ Use domain knowledge to guide extraction
✅ Always validate improvements on test data
✅ Document your feature creation logic
✅ Handle edge cases carefully

Next Steps:
→ Transform features (scaling, normalization)
→ Select most important features
→ Test on multiple models""")